# V2G voltage support on IEEE-34 — reproduction, and an honest test of whether foresight helps

**What this notebook shows, end to end:**

1. **Reproduction sanity** — baseline vs. closed-loop droop (violation-hours), single- & multi-hub, mild & aggressive load. Establishes the identical setup and the central finding: coordinated hubs hold voltage, while a single hub barely moves feeder-wide violations.
2. **The extension (residual-over-droop RL, SOC/time-aware).** Action is `P = clip(droop_P + a·P_rated, 0, P_rated)`, so `a=0` **is** droop — a guaranteed performance floor — and the agent can only *withhold* discharge or *add* support, timed using aggregate SOC / availability / hour-of-day in its observation.
3. **The measured outcome (single-hub): the agent does not beat droop.** At a single hub the worst bus never enters the voltage band, so there is no in-band slack to withhold and the reward collapses to "discharge as hard as possible." Full numbers and diagnosis in **`RESULTS.md`**.
4. **Availability sweep (40–70%)** — no availability level opens a gap at a single hub.

**What this establishes:** single-hub V2G voltage support on this feeder is limited by **energy and siting, not by controller sophistication**. A controller given battery state, availability and time-of-day — floored at droop so it *cannot* underperform by construction — still cannot do better. The lever is coordination across hubs and where the hubs go, not a smarter local policy.

Runs top-to-bottom, self-contained (writes its own feeder + modules). ~15–20 min on CPU.


In [ ]:
# 0. dependencies (installs only what's missing; Kaggle already has torch/numpy/pandas)
import sys, subprocess
def _pipq(*pkgs): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs])
for mod, pkg in [("opendssdirect","opendssdirect.py"),
                 ("gymnasium","gymnasium"),
                 ("stable_baselines3","stable-baselines3")]:
    try: __import__(mod)
    except ImportError: print("installing", pkg); _pipq(pkg)
print("deps OK")

### Materialize the feeder and the tested modules (self-contained)

In [ ]:
%%writefile ieee34_master.dss
! Standard (Mod 1) model of IEEE 34 Bus Test Feeder

! Note: Mod 2 better accounts for distributed load.

Clear
Set DefaultBaseFrequency=60

New object=circuit.ieee34-1
~ basekv=69 pu=1.05 angle=30 mvasc3=200000  !stiffen up a bit over DSS default

! Substation Transformer  -- Modification: Make source very stiff by defining a tiny leakage Z
New Transformer.SubXF Phases=3 Windings=2 Xhl=0.01    ! normally 8
~ wdg=1 bus=sourcebus conn=Delta kv=69    kva=25000   %r=0.0005   !reduce %r, too
~ wdg=2 bus=800       conn=wye   kv=24.9  kva=25000   %r=0.0005

! import line codes with phase impedance matrices
Redirect IEEELineCodes.dss   ! revised according to Later test feeder doc

! Lines
New Line.L1     Phases=3 Bus1=800.1.2.3  Bus2=802.1.2.3  LineCode=300  Length=2.58   units=kft
New Line.L2     Phases=3 Bus1=802.1.2.3  Bus2=806.1.2.3  LineCode=300  Length=1.73   units=kft
New Line.L3     Phases=3 Bus1=806.1.2.3  Bus2=808.1.2.3  LineCode=300  Length=32.23   units=kft
New Line.L4     Phases=1 Bus1=808.2      Bus2=810.2      LineCode=303  Length=5.804   units=kft
New Line.L5     Phases=3 Bus1=808.1.2.3  Bus2=812.1.2.3  LineCode=300  Length=37.5   units=kft
New Line.L6     Phases=3 Bus1=812.1.2.3  Bus2=814.1.2.3  LineCode=300  Length=29.73   units=kft
New Line.L7     Phases=3 Bus1=814r.1.2.3 Bus2=850.1.2.3  LineCode=301  Length=0.01   units=kft
New Line.L8     Phases=1 Bus1=816.1      Bus2=818.1      LineCode=302  Length=1.71   units=kft
New Line.L9     Phases=3 Bus1=816.1.2.3  Bus2=824.1.2.3  LineCode=301  Length=10.21   units=kft
New Line.L10    Phases=1 Bus1=818.1      Bus2=820.1      LineCode=302  Length=48.15   units=kft
New Line.L11    Phases=1 Bus1=820.1      Bus2=822.1      LineCode=302  Length=13.74   units=kft
New Line.L12    Phases=1 Bus1=824.2      Bus2=826.2      LineCode=303  Length=3.03   units=kft
New Line.L13    Phases=3 Bus1=824.1.2.3  Bus2=828.1.2.3  LineCode=301  Length=0.84   units=kft
New Line.L14    Phases=3 Bus1=828.1.2.3  Bus2=830.1.2.3  LineCode=301  Length=20.44   units=kft
New Line.L15    Phases=3 Bus1=830.1.2.3  Bus2=854.1.2.3  LineCode=301  Length=0.52   units=kft
New Line.L16    Phases=3 Bus1=832.1.2.3  Bus2=858.1.2.3  LineCode=301  Length=4.9   units=kft
New Line.L17    Phases=3 Bus1=834.1.2.3  Bus2=860.1.2.3  LineCode=301  Length=2.02   units=kft
New Line.L18    Phases=3 Bus1=834.1.2.3  Bus2=842.1.2.3  LineCode=301  Length=0.28   units=kft
New Line.L19    Phases=3 Bus1=836.1.2.3  Bus2=840.1.2.3  LineCode=301  Length=0.86   units=kft
New Line.L20    Phases=3 Bus1=836.1.2.3  Bus2=862.1.2.3  LineCode=301  Length=0.28   units=kft
New Line.L21    Phases=3 Bus1=842.1.2.3  Bus2=844.1.2.3  LineCode=301  Length=1.35   units=kft
New Line.L22    Phases=3 Bus1=844.1.2.3  Bus2=846.1.2.3  LineCode=301  Length=3.64   units=kft
New Line.L23    Phases=3 Bus1=846.1.2.3  Bus2=848.1.2.3  LineCode=301  Length=0.53   units=kft
New Line.L24    Phases=3 Bus1=850.1.2.3  Bus2=816.1.2.3  LineCode=301  Length=0.31   units=kft
New Line.L25    Phases=3 Bus1=852r.1.2.3 Bus2=832.1.2.3  LineCode=301  Length=0.01   units=kft

! 24.9/4.16 kV  Transformer
New Transformer.XFM1  Phases=3 Windings=2 Xhl=4.08
~ wdg=1 bus=832       conn=wye   kv=24.9  kva=500    %r=0.95
~ wdg=2 bus=888       conn=Wye   kv=4.16  kva=500    %r=0.95

New Line.L26    Phases=1 Bus1=854.2      Bus2=856.2      LineCode=303  Length=23.33   units=kft
New Line.L27    Phases=3 Bus1=854.1.2.3  Bus2=852.1.2.3  LineCode=301  Length=36.83   units=kft
! 9-17-10 858-864 changed to phase A per error report
New Line.L28    Phases=1 Bus1=858.1      Bus2=864.1      LineCode=303  Length=1.62   units=kft
New Line.L29    Phases=3 Bus1=858.1.2.3  Bus2=834.1.2.3  LineCode=301  Length=5.83   units=kft
New Line.L30    Phases=3 Bus1=860.1.2.3  Bus2=836.1.2.3  LineCode=301  Length=2.68   units=kft
New Line.L31    Phases=1 Bus1=862.2      Bus2=838.2      LineCode=304  Length=4.86   units=kft
New Line.L32    Phases=3 Bus1=888.1.2.3  Bus2=890.1.2.3  LineCode=300  Length=10.56   units=kft

! Capacitors
New Capacitor.C844      Bus1=844        Phases=3        kVAR=300        kV=24.9
New Capacitor.C848      Bus1=848        Phases=3        kVAR=450        kV=24.9

! Regulators - three independent phases
! Regulator 1
new transformer.reg1a phases=1 windings=2 bank=reg1 buses=(814.1 814r.1) conns='wye wye' kvs="14.376 14.376" kvas="20000 20000" XHL=1
new regcontrol.creg1a transformer=reg1a winding=2 vreg=122 band=2 ptratio=120 ctprim=100 R=2.7 X=1.6
new transformer.reg1b phases=1 windings=2 bank=reg1 buses=(814.2 814r.2) conns='wye wye' kvs="14.376 14.376" kvas="20000 20000" XHL=1
new regcontrol.creg1b transformer=reg1b winding=2 vreg=122 band=2 ptratio=120 ctprim=100 R=2.7 X=1.6
new transformer.reg1c phases=1 windings=2 bank=reg1 buses=(814.3 814r.3) conns='wye wye' kvs="14.376 14.376" kvas="20000 20000" XHL=1
new regcontrol.creg1c transformer=reg1c winding=2 vreg=122 band=2 ptratio=120 ctprim=100 R=2.7 X=1.6

! Regulator 2
new transformer.reg2a phases=1 windings=2 bank=reg2 buses=(852.1 852r.1) conns='wye wye' kvs="14.376 14.376" kvas="20000 20000" XHL=1
new regcontrol.creg2a transformer=reg2a winding=2 vreg=124 band=2 ptratio=120 ctprim=100 R=2.5 X=1.5
new transformer.reg2b phases=1 windings=2 bank=reg2 buses=(852.2 852r.2) conns='wye wye' kvs="14.376 14.376" kvas="20000 20000" XHL=1
new regcontrol.creg2b transformer=reg2b winding=2 vreg=124 band=2 ptratio=120 ctprim=100 R=2.5 X=1.5
new transformer.reg2c phases=1 windings=2 bank=reg2 buses=(852.3 852r.3) conns='wye wye' kvs="14.376 14.376" kvas="20000 20000" XHL=1
new regcontrol.creg2c transformer=reg2c winding=2 vreg=124 band=2 ptratio=120 ctprim=100 R=2.5 X=1.5

! spot loads
New Load.S860       Bus1=860   Phases=3 Conn=Wye   Model=1 kV= 24.900 kW=  60.0 kVAR=  48.0
New Load.S840       Bus1=840   Phases=3 Conn=Wye   Model=5 kV= 24.900 kW=  27.0 kVAR=  21.0
New Load.S844       Bus1=844   Phases=3 Conn=Wye   Model=2 kV= 24.900 kW= 405.0 kVAR= 315.0

New Load.S848       Bus1=848   Phases=3 Conn=Delta Model=1 kV= 24.900 kW=  60.0 kVAR=  48.0
New Load.S830a      Bus1=830.1.2 Phases=1 Conn=Delta Model=2 kV= 24.900 kW=  10.0 kVAR=   5.0
New Load.S830b      Bus1=830.2.3 Phases=1 Conn=Delta Model=2 kV= 24.900 kW=  10.0 kVAR=   5.0
New Load.S830c      Bus1=830.3.1 Phases=1 Conn=Delta Model=2 kV= 24.900 kW=  25.0 kVAR=  10.0
New Load.S890       Bus1=890   Phases=3 Conn=Delta Model=5 kV=  4.160 kW= 450.0 kVAR= 225.0

! distributed loads
New Load.D802_806sb Bus1=802.2 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=  15.0 kVAR=   7.5
New Load.D802_806rb Bus1=806.2 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=  15.0 kVAR=   7.5
New Load.D802_806sc Bus1=802.3 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=  12.5 kVAR=   7.0
New Load.D802_806rc Bus1=806.3 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=  12.5 kVAR=   7.0

New Load.D808_810sb Bus1=808.2 Phases=1 Conn=Wye   Model=4 kV= 14.376 kW=   8.0 kVAR=   4.0
New Load.D808_810rb Bus1=810.2 Phases=1 Conn=Wye   Model=4 kV= 14.376 kW=   8.0 kVAR=   4.0

New Load.D818_820sa Bus1=818.1 Phases=1 Conn=Wye   Model=2 kV= 14.376 kW=  17.0 kVAR=   8.5
New Load.D818_820ra Bus1=820.1 Phases=1 Conn=Wye   Model=2 kV= 14.376 kW=  17.0 kVAR=   8.5

New Load.D820_822sa Bus1=820.1 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=  67.5 kVAR=  35.0
New Load.D820_822ra Bus1=822.1 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=  67.5 kVAR=  35.0

New Load.D816_824sb Bus1=816.2.3 Phases=1 Conn=Delta Model=5 kV= 24.900 kW=   2.5 kVAR=   1.0
New Load.D816_824rb Bus1=824.2.3 Phases=1 Conn=Delta Model=5 kV= 24.900 kW=   2.5 kVAR=   1.0

New Load.D824_826sb Bus1=824.2 Phases=1 Conn=Wye   Model=5 kV= 14.376 kW=  20.0 kVAR=  10.0
New Load.D824_826rb Bus1=826.2 Phases=1 Conn=Wye   Model=5 kV= 14.376 kW=  20.0 kVAR=  10.0
New Load.D824_828sc Bus1=824.3 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=   2.0 kVAR=   1.0
New Load.D824_828rc Bus1=828.3 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=   2.0 kVAR=   1.0

New Load.D828_830sa Bus1=828.1 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=   3.5 kVAR=   1.5
New Load.D828_830ra Bus1=830.1 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=   3.5 kVAR=   1.5

New Load.D854_856sb Bus1=854.2 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=   2.0 kVAR=   1.0
New Load.D854_856rb Bus1=856.2 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=   2.0 kVAR=   1.0

New Load.D832_858sa Bus1=832.1 Phases=1 Conn=Delta Model=2 kV= 24.900 kW=   3.5 kVAR=   1.5
New Load.D832_858ra Bus1=858.1 Phases=1 Conn=Delta Model=2 kV= 24.900 kW=   3.5 kVAR=   1.5
New Load.D832_858sb Bus1=832.2 Phases=1 Conn=Delta Model=2 kV= 24.900 kW=   1.0 kVAR=   0.5
New Load.D832_858rb Bus1=858.2 Phases=1 Conn=Delta Model=2 kV= 24.900 kW=   1.0 kVAR=   0.5
New Load.D832_858sc Bus1=832.3 Phases=1 Conn=Delta Model=2 kV= 24.900 kW=   3.0 kVAR=   1.5
New Load.D832_858rc Bus1=858.3 Phases=1 Conn=Delta Model=2 kV= 24.900 kW=   3.0 kVAR=   1.5

! 9-17-10 858-864 changed to phase A per error report
New Load.D858_864sb Bus1=858.1 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=   1.0 kVAR=   0.5
New Load.D858_864rb Bus1=864.1 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=   1.0 kVAR=   0.5

New Load.D858_834sa Bus1=858.1.2 Phases=1 Conn=Delta Model=1 kV= 24.900 kW=   2.0 kVAR=   1.0
New Load.D858_834ra Bus1=834.1.2 Phases=1 Conn=Delta Model=1 kV= 24.900 kW=   2.0 kVAR=   1.0
New Load.D858_834sb Bus1=858.2.3 Phases=1 Conn=Delta Model=1 kV= 24.900 kW=   7.5 kVAR=   4.0
New Load.D858_834rb Bus1=834.2.3 Phases=1 Conn=Delta Model=1 kV= 24.900 kW=   7.5 kVAR=   4.0
New Load.D858_834sc Bus1=858.3.1 Phases=1 Conn=Delta Model=1 kV= 24.900 kW=   6.5 kVAR=   3.5
New Load.D858_834rc Bus1=834.3.1 Phases=1 Conn=Delta Model=1 kV= 24.900 kW=   6.5 kVAR=   3.5

New Load.D834_860sa Bus1=834.1.2 Phases=1 Conn=Delta Model=2 kV= 24.900 kW=   8.0 kVAR=   4.0
New Load.D834_860ra Bus1=860.1.2 Phases=1 Conn=Delta Model=2 kV= 24.900 kW=   8.0 kVAR=   4.0
New Load.D834_860sb Bus1=834.2.3 Phases=1 Conn=Delta Model=2 kV= 24.900 kW=  10.0 kVAR=   5.0
New Load.D834_860rb Bus1=860.2.3 Phases=1 Conn=Delta Model=2 kV= 24.900 kW=  10.0 kVAR=   5.0
New Load.D834_860sc Bus1=834.3.1 Phases=1 Conn=Delta Model=2 kV= 24.900 kW=  55.0 kVAR=  27.5
New Load.D834_860rc Bus1=860.3.1 Phases=1 Conn=Delta Model=2 kV= 24.900 kW=  55.0 kVAR=  27.5

New Load.D860_836sa Bus1=860.1.2 Phases=1 Conn=Delta Model=1 kV= 24.900 kW=  15.0 kVAR=   7.5
New Load.D860_836ra Bus1=836.1.2 Phases=1 Conn=Delta Model=1 kV= 24.900 kW=  15.0 kVAR=   7.5
New Load.D860_836sb Bus1=860.2.3 Phases=1 Conn=Delta Model=1 kV= 24.900 kW=   5.0 kVAR=   3.0
New Load.D860_836rb Bus1=836.2.3 Phases=1 Conn=Delta Model=1 kV= 24.900 kW=   5.0 kVAR=   3.0
New Load.D860_836sc Bus1=860.3.1 Phases=1 Conn=Delta Model=1 kV= 24.900 kW=  21.0 kVAR=  11.0
New Load.D860_836rc Bus1=836.3.1 Phases=1 Conn=Delta Model=1 kV= 24.900 kW=  21.0 kVAR=  11.0

New Load.D836_840sa Bus1=836.1.2 Phases=1 Conn=Delta Model=5 kV= 24.900 kW=   9.0 kVAR=   4.5
New Load.D836_840ra Bus1=840.1.2 Phases=1 Conn=Delta Model=5 kV= 24.900 kW=   9.0 kVAR=   4.5
New Load.D836_840sb Bus1=836.2.3 Phases=1 Conn=Delta Model=5 kV= 24.900 kW=  11.0 kVAR=   5.5
New Load.D836_840rb Bus1=840.2.3 Phases=1 Conn=Delta Model=5 kV= 24.900 kW=  11.0 kVAR=   5.5

New Load.D862_838sb Bus1=862.2 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=  14.0 kVAR=   7.0
New Load.D862_838rb Bus1=838.2 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=  14.0 kVAR=   7.0

New Load.D842_844sa Bus1=842.1 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=   4.5 kVAR=   2.5
New Load.D842_844ra Bus1=844.1 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=   4.5 kVAR=   2.5

New Load.D844_846sb Bus1=844.2 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=  12.5 kVAR=   6.0
New Load.D844_846rb Bus1=846.2 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=  12.5 kVAR=   6.0
New Load.D844_846sc Bus1=844.3 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=  10.0 kVAR=   5.5
New Load.D844_846rc Bus1=846.3 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=  10.0 kVAR=   5.5

New Load.D846_848sb Bus1=846.2 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=  11.5 kVAR=   5.5
New Load.D846_848rb Bus1=848.2 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=  11.5 kVAR=   5.5

! Script to revise Vminpu property on all loads to allow voltage to sag to 85% without switching
! to constant Z model
Load.s860.vminpu=.85
Load.s840.vminpu=.85
Load.s844.vminpu=.85
Load.s848.vminpu=.85
Load.s830a.vminpu=.85
Load.s830b.vminpu=.85
Load.s830c.vminpu=.85
Load.s890.vminpu=.85
Load.d802_806sb.vminpu=.85
Load.d802_806rb.vminpu=.85
Load.d802_806sc.vminpu=.85
Load.d802_806rc.vminpu=.85
Load.d808_810sb.vminpu=.85
Load.d808_810rb.vminpu=.85
Load.d818_820sa.vminpu=.85
Load.d818_820ra.vminpu=.85
Load.d820_822sa.vminpu=.85
Load.d820_822ra.vminpu=.85
Load.d816_824sb.vminpu=.85
Load.d816_824rb.vminpu=.85
Load.d824_826sb.vminpu=.85
Load.d824_826rb.vminpu=.85
Load.d824_828sc.vminpu=.85
Load.d824_828rc.vminpu=.85
Load.d828_830sa.vminpu=.85
Load.d828_830ra.vminpu=.85
Load.d854_856sb.vminpu=.85
Load.d854_856rb.vminpu=.85
Load.d832_858sa.vminpu=.85
Load.d832_858ra.vminpu=.85
Load.d832_858sb.vminpu=.85
Load.d832_858rb.vminpu=.85
Load.d832_858sc.vminpu=.85
Load.d832_858rc.vminpu=.85
Load.d858_864sb.vminpu=.85
Load.d858_864rb.vminpu=.85
Load.d858_834sa.vminpu=.85
Load.d858_834ra.vminpu=.85
Load.d858_834sb.vminpu=.85
Load.d858_834rb.vminpu=.85
Load.d858_834sc.vminpu=.85
Load.d858_834rc.vminpu=.85
Load.d834_860sa.vminpu=.85
Load.d834_860ra.vminpu=.85
Load.d834_860sb.vminpu=.85
Load.d834_860rb.vminpu=.85
Load.d834_860sc.vminpu=.85
Load.d834_860rc.vminpu=.85
Load.d860_836sa.vminpu=.85
Load.d860_836ra.vminpu=.85
Load.d860_836sb.vminpu=.85
Load.d860_836rb.vminpu=.85
Load.d860_836sc.vminpu=.85
Load.d860_836rc.vminpu=.85
Load.d836_840sa.vminpu=.85
Load.d836_840ra.vminpu=.85
Load.d836_840sb.vminpu=.85
Load.d836_840rb.vminpu=.85
Load.d862_838sb.vminpu=.85
Load.d862_838rb.vminpu=.85
Load.d842_844sa.vminpu=.85
Load.d842_844ra.vminpu=.85
Load.d844_846sb.vminpu=.85
Load.d844_846rb.vminpu=.85
Load.d844_846sc.vminpu=.85
Load.d844_846rc.vminpu=.85
Load.d846_848sb.vminpu=.85
Load.d846_848rb.vminpu=.85


! let the DSS estimate voltage bases automatically
Set VoltageBases = "69,24.9,4.16, .48"
CalcVoltageBases


In [ ]:
%%writefile IEEELineCodes.dss
! this file was corrected 9/16/2010 to match the values in Kersting's files



! These line codes are used in the 123-bus circuit

New linecode.1 nphases=3 BaseFreq=60 units=kft
!!!~ rmatrix = (0.088205 | 0.0312137 0.0901946 | 0.0306264 0.0316143 0.0889665 )
!!!~ xmatrix = (0.20744 | 0.0935314 0.200783 | 0.0760312 0.0855879 0.204877 )
!!!~ cmatrix = (2.90301 | -0.679335 3.15896 | -0.22313 -0.481416 2.8965 )
~ rmatrix = [0.086666667 | 0.029545455 0.088371212 | 0.02907197 0.029924242 0.087405303]
~ xmatrix = [0.204166667 | 0.095018939 0.198522727 | 0.072897727 0.080227273 0.201723485]
~ cmatrix = [2.851710072 | -0.920293787  3.004631862 | -0.350755566  -0.585011253 2.71134756]

New linecode.2 nphases=3 BaseFreq=60 units=kft
!!!~ rmatrix = (0.0901946 | 0.0316143 0.0889665 | 0.0312137 0.0306264 0.088205 )
!!!~ xmatrix = (0.200783 | 0.0855879 0.204877 | 0.0935314 0.0760312 0.20744 )
!!!~ cmatrix = (3.15896 | -0.481416 2.8965 | -0.679335 -0.22313 2.90301 )
~ rmatrix = [0.088371212 | 0.02992424  0.087405303 | 0.029545455 0.02907197 0.086666667]
~ xmatrix = [0.198522727 | 0.080227273  0.201723485 | 0.095018939 0.072897727 0.204166667]
~ cmatrix = [3.004631862 | -0.585011253 2.71134756 | -0.920293787  -0.350755566  2.851710072]

New linecode.3 nphases=3 BaseFreq=60 units=kft
!!!~ rmatrix = (0.0889665 | 0.0306264 0.088205 | 0.0316143 0.0312137 0.0901946 )
!!!~ xmatrix = (0.204877 | 0.0760312 0.20744 | 0.0855879 0.0935314 0.200783 )
!!!~ cmatrix = (2.8965 | -0.22313 2.90301 | -0.481416 -0.679335 3.15896 )

~ rmatrix = [0.087405303 | 0.02907197 0.086666667  | 0.029924242 0.029545455 0.088371212]
~ xmatrix = [0.201723485 | 0.072897727 0.204166667 | 0.080227273 0.095018939 0.198522727]
~ cmatrix = [2.71134756  | -0.350755566 2.851710072 | -0.585011253 -0.920293787 3.004631862]

New linecode.4 nphases=3 BaseFreq=60 units=kft
!!!~ rmatrix = (0.0889665 | 0.0316143 0.0901946 | 0.0306264 0.0312137 0.088205 )
!!!~ xmatrix = (0.204877 | 0.0855879 0.200783 | 0.0760312 0.0935314 0.20744 )
!!!~ cmatrix = (2.8965 | -0.481416 3.15896 | -0.22313 -0.679335 2.90301 )
~ rmatrix = [0.087405303 | 0.029924242 0.088371212 | 0.02907197   0.029545455 0.086666667]
~ xmatrix = [0.201723485 | 0.080227273 0.198522727 | 0.072897727 0.095018939 0.204166667]
~ cmatrix = [2.71134756  | -0.585011253 3.004631862 | -0.350755566 -0.920293787 2.851710072]

New linecode.5 nphases=3 BaseFreq=60 units=kft
!!!~ rmatrix = (0.0901946 | 0.0312137 0.088205 | 0.0316143 0.0306264 0.0889665 )
!!!~ xmatrix = (0.200783 | 0.0935314 0.20744 | 0.0855879 0.0760312 0.204877 )
!!!~ cmatrix = (3.15896 | -0.679335 2.90301 | -0.481416 -0.22313 2.8965 )

~ rmatrix = [0.088371212  |  0.029545455  0.086666667  |  0.029924242  0.02907197  0.087405303]
~ xmatrix = [0.198522727  |  0.095018939  0.204166667  |  0.080227273  0.072897727  0.201723485]
~ cmatrix = [3.004631862  | -0.920293787  2.851710072  |  -0.585011253  -0.350755566  2.71134756]

New linecode.6 nphases=3 BaseFreq=60 units=kft
!!!~ rmatrix = (0.088205 | 0.0306264 0.0889665 | 0.0312137 0.0316143 0.0901946 )
!!!~ xmatrix = (0.20744 | 0.0760312 0.204877 | 0.0935314 0.0855879 0.200783 )
!!!~ cmatrix = (2.90301 | -0.22313 2.8965 | -0.679335 -0.481416 3.15896 )
~ rmatrix = [0.086666667 | 0.02907197  0.087405303 | 0.029545455  0.029924242  0.088371212]
~ xmatrix = [0.204166667 | 0.072897727  0.201723485 | 0.095018939  0.080227273  0.198522727]
~ cmatrix = [2.851710072 | -0.350755566  2.71134756 | -0.920293787  -0.585011253  3.004631862]
New linecode.7 nphases=2 BaseFreq=60 units=kft
!!!~ rmatrix = (0.088205 | 0.0306264 0.0889665 )
!!!~ xmatrix = (0.20744 | 0.0760312 0.204877 )
!!!~ cmatrix = (2.75692 | -0.326659 2.82313 )
~ rmatrix = [0.086666667 | 0.02907197  0.087405303]
~ xmatrix = [0.204166667 | 0.072897727  0.201723485]
~ cmatrix = [2.569829596 | -0.52995137  2.597460011]
New linecode.8 nphases=2 BaseFreq=60 units=kft
!!!~ rmatrix = (0.088205 | 0.0306264 0.0889665 )
!!!~ xmatrix = (0.20744 | 0.0760312 0.204877 )
!!!~ cmatrix = (2.75692 | -0.326659 2.82313 )
~ rmatrix = [0.086666667 | 0.02907197  0.087405303]
~ xmatrix = [0.204166667 | 0.072897727  0.201723485]
~ cmatrix = [2.569829596 | -0.52995137  2.597460011]
New linecode.9 nphases=1 BaseFreq=60 units=kft
!!!~ rmatrix = (0.254428 )
!!!~ xmatrix = (0.259546 )
!!!~ cmatrix = (2.50575 )
~ rmatrix = [0.251742424]
~ xmatrix = [0.255208333]
~ cmatrix = [2.270366128]
New linecode.10 nphases=1 BaseFreq=60 units=kft
!!!~ rmatrix = (0.254428 )
!!!~ xmatrix = (0.259546 )
!!!~ cmatrix = (2.50575 )
~ rmatrix = [0.251742424]
~ xmatrix = [0.255208333]
~ cmatrix = [2.270366128]
New linecode.11 nphases=1 BaseFreq=60 units=kft
!!!~ rmatrix = (0.254428 )
!!!~ xmatrix = (0.259546 )
!!!~ cmatrix = (2.50575 )
~ rmatrix = [0.251742424]
~ xmatrix = [0.255208333]
~ cmatrix = [2.270366128]
New linecode.12 nphases=3 BaseFreq=60 units=kft
!!!~ rmatrix = (0.291814 | 0.101656 0.294012 | 0.096494 0.101656 0.291814 )
!!!~ xmatrix = (0.141848 | 0.0517936 0.13483 | 0.0401881 0.0517936 0.141848 )
!!!~ cmatrix = (53.4924 | 0 53.4924 | 0 0 53.4924 )
~ rmatrix = [0.288049242 | 0.09844697  0.29032197 | 0.093257576  0.09844697  0.288049242]
~ xmatrix = [0.142443182 | 0.052556818  0.135643939 | 0.040852273  0.052556818  0.142443182]
~ cmatrix = [33.77150149 | 0  33.77150149 | 0  0  33.77150149]

! These line codes are used in the 34-node test feeder

New linecode.300 nphases=3 basefreq=60   units=kft   ! ohms per 1000ft  Corrected 11/30/05
~ rmatrix = [0.253181818   |  0.039791667     0.250719697  |   0.040340909      0.039128788     0.251780303]  !ABC ORDER
~ xmatrix = [0.252708333   |  0.109450758     0.256988636  |   0.094981061      0.086950758     0.255132576]
~ CMATRIX = [2.680150309   | -0.769281006     2.5610381    |  -0.499507676     -0.312072984     2.455590387]
New linecode.301 nphases=3 basefreq=60   units=kft
~ rmatrix = [0.365530303   |   0.04407197      0.36282197   |   0.04467803       0.043333333     0.363996212]
~ xmatrix = [0.267329545   |   0.122007576     0.270473485  |   0.107784091      0.099204545     0.269109848] 
~ cmatrix = [2.572492163   |  -0.72160598      2.464381882  |  -0.472329395     -0.298961096     2.368881119]
New linecode.302 nphases=1 basefreq=60   units=kft
~ rmatrix = (0.530208 )
~ xmatrix = (0.281345 )
~ cmatrix = (2.12257 )
New linecode.303 nphases=1 basefreq=60   units=kft
~ rmatrix = (0.530208 )
~ xmatrix = (0.281345 )
~ cmatrix = (2.12257 )
New linecode.304 nphases=1 basefreq=60   units=kft
~ rmatrix = (0.363958 )
~ xmatrix = (0.269167 )
~ cmatrix = (2.1922 )


! This may be for the 4-node test feeder, but is not actually referenced.
!  instead, the 4Bus*.dss files all use the wiredata and linegeometry inputs
!  to calculate these matrices from physical data.

New linecode.400 nphases=3 BaseFreq=60
~ rmatrix = (0.088205 | 0.0312137 0.0901946 | 0.0306264 0.0316143 0.0889665 )
~ xmatrix = (0.20744 | 0.0935314 0.200783 | 0.0760312 0.0855879 0.204877 )
~ cmatrix = (2.90301 | -0.679335 3.15896 | -0.22313 -0.481416 2.8965 )

! These are for the 13-node test feeder

New linecode.601 nphases=3 BaseFreq=60
!!!~ rmatrix = (0.0674673 | 0.0312137 0.0654777 | 0.0316143 0.0306264 0.0662392 )
!!!~ xmatrix = (0.195204  | 0.0935314 0.201861 | 0.0855879 0.0760312 0.199298 )
!!!~ cmatrix = (3.32591   | -0.743055 3.04217 | -0.525237 -0.238111 3.03116 )
~ rmatrix = [0.065625    | 0.029545455  0.063920455  | 0.029924242  0.02907197  0.064659091]
~ xmatrix = [0.192784091 | 0.095018939  0.19844697   | 0.080227273  0.072897727  0.195984848]
~ cmatrix = [3.164838036 | -1.002632425  2.993981593 | -0.632736516  -0.372608713  2.832670203]
New linecode.602 nphases=3 BaseFreq=60
!!!~ rmatrix = (0.144361 | 0.0316143 0.143133 | 0.0312137 0.0306264 0.142372 )
!!!~ xmatrix = (0.226028 | 0.0855879 0.230122 | 0.0935314 0.0760312 0.232686 )
!!!~ cmatrix = (3.01091  | -0.443561 2.77543  | -0.624494 -0.209615 2.77847 )
~ rmatrix = [0.142537879 | 0.029924242  0.14157197   | 0.029545455  0.02907197  0.140833333]
~ xmatrix = [0.22375     | 0.080227273  0.226950758  | 0.095018939  0.072897727  0.229393939]
~ cmatrix = [2.863013423 | -0.543414918  2.602031589 | -0.8492585  -0.330962141  2.725162768]
New linecode.603 nphases=2 BaseFreq=60
!!!~ rmatrix = (0.254472 | 0.0417943 0.253371 )
!!!~ xmatrix = (0.259467 | 0.0912376 0.261431 )
!!!~ cmatrix = (2.54676  | -0.28882 2.49502 )
~ rmatrix = [0.251780303 | 0.039128788  0.250719697]
~ xmatrix = [0.255132576 | 0.086950758  0.256988636]
~ cmatrix = [2.366017603 | -0.452083836  2.343963508]
New linecode.604 nphases=2 BaseFreq=60
!!!~ rmatrix = (0.253371 | 0.0417943 0.254472 )
!!!~ xmatrix = (0.261431 | 0.0912376 0.259467 )
!!!~ cmatrix = (2.49502 | -0.28882 2.54676 )
~ rmatrix = [0.250719697 | 0.039128788   0.251780303]
~ xmatrix = [0.256988636  | 0.086950758  0.255132576]
~ cmatrix = [2.343963508 | -0.452083836 2.366017603]
New linecode.605 nphases=1 BaseFreq=60
!!!~ rmatrix = (0.254428 )
!!!~ xmatrix = (0.259546 )
!!!~ cmatrix = (2.50575 )
~ rmatrix = [0.251742424]
~ xmatrix = [0.255208333]
~ cmatrix = [2.270366128]
New linecode.606 nphases=3 BaseFreq=60
!!!~ rmatrix = (0.152193 | 0.0611362 0.15035 | 0.0546992 0.0611362 0.152193 )
!!!~ xmatrix = (0.0825685 | 0.00548281 0.0745027 | -0.00339824 0.00548281 0.0825685 )
!!!~ cmatrix = (72.7203 | 0 72.7203 | 0 0 72.7203 )
~ rmatrix = [0.151174242 | 0.060454545  0.149450758 | 0.053958333  0.060454545  0.151174242]
~ xmatrix = [0.084526515 | 0.006212121  0.076534091 | -0.002708333  0.006212121  0.084526515]
~ cmatrix = [48.67459408 | 0  48.67459408 | 0  0  48.67459408]
New linecode.607 nphases=1 BaseFreq=60
!!!~ rmatrix = (0.255799 )
!!!~ xmatrix = (0.092284 )
!!!~ cmatrix = (50.7067 )
~ rmatrix = [0.254261364]
~ xmatrix = [0.097045455]
~ cmatrix = [44.70661522]

! These are for the 37-node test feeder, all underground

New linecode.721 nphases=3 BaseFreq=60
!!!~ rmatrix = (0.0554906 | 0.0127467 0.0501597 | 0.00640446 0.0127467 0.0554906 )
!!!~ xmatrix = (0.0372331 | -0.00704588 0.0358645 | -0.00796424 -0.00704588 0.0372331 )
!!!~ cmatrix = (124.851 | 0 124.851 | 0 0 124.851 )
~ rmatrix = [0.055416667 | 0.012746212  0.050113636  | 0.006382576  0.012746212  0.055416667]
~ xmatrix = [0.037367424 | -0.006969697  0.035984848 | -0.007897727  -0.006969697  0.037367424]
~ cmatrix = [80.27484728 | 0  80.27484728            | 0  0  80.27484728]
New linecode.722 nphases=3 BaseFreq=60
!!!~ rmatrix = (0.0902251 | 0.0309584 0.0851482 | 0.0234946 0.0309584 0.0902251 )
!!!~ xmatrix = (0.055991 | -0.00646552 0.0504025 | -0.0117669 -0.00646552 0.055991 )
!!!~ cmatrix = (93.4896 | 0 93.4896 | 0 0 93.4896 )
~ rmatrix = [0.089981061 | 0.030852273  0.085        | 0.023371212  0.030852273  0.089981061]
~ xmatrix = [0.056306818 | -0.006174242  0.050719697 | -0.011496212  -0.006174242  0.056306818]
~ cmatrix = [64.2184109 | 0  64.2184109              | 0  0  64.2184109]
New linecode.723 nphases=3 BaseFreq=60
!!!~ rmatrix = (0.247572 | 0.0947678 0.249104 | 0.0893782 0.0947678 0.247572 )
!!!~ xmatrix = (0.126339 | 0.0390337 0.118816 | 0.0279344 0.0390337 0.126339 )
!!!~ cmatrix = (58.108 | 0 58.108 | 0 0 58.108 )
~ rmatrix = [0.245 | 0.092253788  0.246628788 | 0.086837121  0.092253788  0.245]
~ xmatrix = [0.127140152 | 0.039981061  0.119810606 | 0.028806818  0.039981061  0.127140152]
~ cmatrix = [37.5977112 | 0  37.5977112 | 0  0  37.5977112]
New linecode.724 nphases=3 BaseFreq=60
!!!~ rmatrix = (0.399883 | 0.101765 0.402011 | 0.0965199 0.101765 0.399883 )
!!!~ xmatrix = (0.146325 | 0.0510963 0.139305 | 0.0395402 0.0510963 0.146325 )
!!!~ cmatrix = (46.9685 | 0 46.9685 | 0 0 46.9685 )
~ rmatrix = [0.396818182 | 0.098560606  0.399015152 | 0.093295455  0.098560606  0.396818182]
~ xmatrix = [0.146931818 | 0.051856061  0.140113636 | 0.040208333  0.051856061  0.146931818]
~ cmatrix = [30.26701029 | 0  30.26701029 | 0  0  30.26701029]


In [ ]:
%%writefile v2g_core.py
"""
V2G voltage-regulation reproduction — CORE (deterministic parts).
Tests everything except SAC (feeder wrapper, EV fleet, droop baseline, daily eval).
"""
import os, numpy as np

MASTER = os.path.abspath("ieee34_master.dss")

# ----------------------------- CONFIG -----------------------------
CFG = dict(
    v_min=0.95, v_max=1.05,
    hub_buses_multi=["890", "844", "832", "830", "860"],
    hub_bus_single=["890"],
    P_rated=500.0, Q_rated=400.0,          # kW, kVAr per hub
    ev_capacity=75.0, soc_init=0.7, soc_min=0.2, soc_max=0.9,
    soh=0.95, eta_inv=0.96, c_rate=0.5, n_ev=15,
    active_hours=list(range(6, 24)),       # 6:00 .. 23:00
    peak_mild=1.5, peak_aggr=3.0,
)

# normalized daily load shape (peak=1.0 at ~18:00) and EV availability
LOAD_SHAPE = np.array([0.24,0.22,0.20,0.20,0.22,0.26,0.30,0.42,0.55,0.63,
                       0.70,0.74,0.77,0.79,0.82,0.86,0.92,0.97,1.00,0.96,
                       0.86,0.66,0.44,0.30])
AVAIL = np.array([0.85,0.85,0.85,0.85,0.80,0.75,0.65,0.55,0.50,0.47,
                  0.45,0.45,0.48,0.50,0.55,0.60,0.65,0.72,0.78,0.82,
                  0.85,0.85,0.85,0.85])

def lam_profile(peak):  return LOAD_SHAPE * peak

# ----------------------------- FEEDER -----------------------------
import opendssdirect as dss

# IEEE-34 line-to-line voltage bases (kV). Bus 890 (and 888) sit on the 4.16 kV
# transformer secondary; every other hub bus is on the 24.9 kV main feeder.
# Used as a fallback because some opendssdirect builds (e.g. the NumPy-2.0 wheel
# on Kaggle) report Bus.kVBase()=0 right after compile, which would create
# malformed kv=0 generators and corrupt the multi-hub results.
BASE_LL_KV = {"890": 4.16, "888": 4.16}

class Feeder:
    def __init__(self, hub_buses):
        self.hub_buses = hub_buses
        dss.Command("Clear"); dss.Command(f'Compile "{MASTER}"')
        dss.Command("Set ControlMode=OFF")          # regulators frozen (matches paper)
        dss.Command("Set MaxIterations=100")        # help PF converge under heavy PQ injection
        dss.Command("CalcVoltageBases")             # compute per-unit bases before we read them
        dss.Command("Solve")
        self.buses = [b for b in dss.Circuit.AllBusNames() if b.lower() != "sourcebus"]
        # add a controllable 3-phase generator at each hub bus (kv = bus base LL)
        self.hub_kv = {}
        for b in hub_buses:
            dss.Circuit.SetActiveBus(b)
            kvb = dss.Bus.kVBase()
            kv = round(kvb * np.sqrt(3), 3) if kvb > 0.1 else BASE_LL_KV.get(b.lower(), 24.9)
            self.hub_kv[b] = kv
            dss.Command(f"New Generator.hub{b} bus1={b}.1.2.3 phases=3 kv={kv} "
                        f"kw=0 kvar=0 model=1 Vminpu=0.5 Vmaxpu=1.5 status=fixed")
        dss.Command("Solve")

    def set_load(self, lam):
        dss.Command(f"Set LoadMult={lam}")

    def set_hub(self, bus, p_kw, q_kvar):
        dss.Command(f"Generator.hub{bus}.kW={p_kw}")
        dss.Command(f"Generator.hub{bus}.kvar={q_kvar}")

    def zero_hubs(self):
        for b in self.hub_buses: self.set_hub(b, 0.0, 0.0)

    def solve(self):
        dss.Command("Solve")
        return dss.Solution.Converged()

    def bus_vpu(self):
        """per-bus mean p.u. voltage over energized phases, ordered like self.buses"""
        out = np.empty(len(self.buses))
        for i, b in enumerate(self.buses):
            dss.Circuit.SetActiveBus(b)
            vs = [v for v in dss.Bus.puVmagAngle()[0::2] if v > 0.01]
            out[i] = np.mean(vs) if vs else 1.0
        return out

    def hub_vpu(self, bus):
        dss.Circuit.SetActiveBus(bus)
        vs = [v for v in dss.Bus.puVmagAngle()[0::2] if v > 0.01]
        return float(np.mean(vs)) if vs else 1.0

    def feeder_mean(self):
        return float(np.mean(self.bus_vpu()))

def reward_from_v(vpu, vmin=0.95, vmax=1.05):
    inb = np.all((vpu >= vmin) & (vpu <= vmax))
    Rvb = 10.0 if inb else 0.0
    pen = np.where(vpu < vmin, (vmin - vpu) * 100,
          np.where(vpu > vmax, (vpu - vmax) * 100, 0.0)).sum()
    return Rvb - pen

# ----------------------------- EV FLEET -----------------------------
class EVFleet:
    """Aggregate per-hub fleet with SOC/availability-limited dischargeable power."""
    def __init__(self, cfg):
        self.c = cfg; self.soc = cfg["soc_init"]; self.soh = cfg["soh"]
    def reset(self): self.soc = self.c["soc_init"]
    def n_avail(self, hour):
        return max(1, int(round(self.c["n_ev"] * AVAIL[hour])))
    def avail_power(self, hour):
        """max discharge power (kW) this 1-h step, limited by C-rate and usable SOC energy"""
        n = self.n_avail(hour); cap, soh = self.c["ev_capacity"], self.soh
        p_crate = n * self.c["c_rate"] * cap                       # kW
        e_usable = n * max(0.0, self.soc - self.c["soc_min"]) * cap * soh  # kWh over 1h -> kW
        return min(p_crate, e_usable)
    def apply(self, p_grid, q_grid, hour):
        """scale requested (p,q) to fleet capability; update SOC. Returns (p_sup,q_sup,rho,n)."""
        s_req = np.hypot(p_grid, q_grid)
        p_fleet = s_req / self.c["eta_inv"]
        p_avail = self.avail_power(hour)
        rho = min(1.0, p_avail / p_fleet) if p_fleet > 1e-6 else 1.0
        p_sup, q_sup = rho * p_grid, rho * q_grid
        # SOC update from active battery energy (discharge drains). dt = 1 h
        n = self.n_avail(hour); cap_tot = n * self.c["ev_capacity"] * self.soh
        p_batt = abs(p_sup) / self.c["eta_inv"] if p_sup >= 0 else abs(p_sup) * self.c["eta_inv"]
        dsoc = -(p_batt * 1.0) / cap_tot if p_sup >= 0 else (p_batt * 1.0) / cap_tot
        self.soc = float(np.clip(self.soc + dsoc, self.c["soc_min"], self.c["soc_max"]))
        return p_sup, q_sup, rho, n

# ----------------------------- DROOP -----------------------------
def droop_pq(v, P_rated, Q_rated, db=0.02, sat_lo=0.94, sat_hi=1.06):
    """symmetric Volt-Watt / Volt-Var: discharge+inject Q when low, charge+absorb Q when high."""
    def frac(v):
        if v < 1 - db:  return min(1.0, (1 - db - v) / (1 - db - sat_lo))   # support (positive)
        if v > 1 + db:  return -min(1.0, (v - (1 + db)) / (sat_hi - (1 + db)))  # curtail/charge
        return 0.0
    f = frac(v)
    return f * P_rated, f * Q_rated

# ----------------------------- DAILY EVALUATION -----------------------------
def evaluate_day(feeder, peak, controller, ev_constrained=False, fleets=None,
                 policy=None, cfg=CFG):
    """controller in {'baseline','droop','rl'}; returns dict of hourly series + stats."""
    lam = lam_profile(peak)
    hours = cfg["active_hours"]
    rec = {"hour": [], "vmean": [], "soc": [], "n_ev": [], "rho": []}
    if fleets is not None:
        for fl in fleets.values(): fl.reset()
    for h in hours:
        feeder.set_load(lam[h]); feeder.zero_hubs(); feeder.solve()
        obs_v = feeder.bus_vpu()
        # ---- compute per-hub setpoints ----
        setpoints = {}
        if controller == "baseline":
            for b in feeder.hub_buses: setpoints[b] = (0.0, 0.0)
        elif controller == "droop":
            setpoints = {b: (0.0, 0.0) for b in feeder.hub_buses}
            damp = 0.6
            for _ in range(25):  # damped fixed-point iterations for stable droop equilibrium
                for b in feeder.hub_buses:
                    v = feeder.hub_vpu(b)
                    p, q = droop_pq(v, cfg["P_rated"], cfg["Q_rated"])
                    p0, q0 = setpoints[b]
                    setpoints[b] = ((1-damp)*p0 + damp*p, (1-damp)*q0 + damp*q)
                _apply_setpoints(feeder, setpoints, ev_constrained, fleets, h, commit=False)
                feeder.solve()
        elif controller == "rl":
            lam_norm = (lam[h] - 0.1) / (4.0 - 0.1)
            obs = np.concatenate([obs_v, [lam_norm]]).astype(np.float32)
            act, _ = policy.predict(obs, deterministic=True)
            for i, b in enumerate(feeder.hub_buses):
                setpoints[b] = (float(act[2*i]) * cfg["P_rated"],
                                float(act[2*i+1]) * cfg["Q_rated"])
        # ---- apply (with fleet scaling + SOC commit) and solve ----
        info = _apply_setpoints(feeder, setpoints, ev_constrained, fleets, h, commit=True)
        feeder.solve()
        vmean = feeder.feeder_mean()
        rec["hour"].append(h); rec["vmean"].append(vmean)
        rec["soc"].append(np.mean([fleets[b].soc for b in feeder.hub_buses]) if fleets else np.nan)
        rec["n_ev"].append(info.get("n", np.nan)); rec["rho"].append(info.get("rho", 1.0))
    vm = np.array(rec["vmean"])
    stats = dict(Mean=round(vm.mean(),3), Min=round(vm.min(),3), Max=round(vm.max(),3),
                 Viol=int((vm < cfg["v_min"]).sum()))
    return stats, rec

def _apply_setpoints(feeder, setpoints, ev_constrained, fleets, hour, commit):
    n_last, rho_last = np.nan, 1.0
    for b, (p, q) in setpoints.items():
        if ev_constrained and fleets is not None:
            if commit:
                p, q, rho, n = fleets[b].apply(p, q, hour); rho_last, n_last = rho, n
            else:
                # preview scaling without committing SOC (for droop iterations)
                s = np.hypot(p, q); pf = s / feeder_eta(); pa = fleets[b].avail_power(hour)
                rho = min(1.0, pa/pf) if pf > 1e-6 else 1.0; p, q = rho*p, rho*q
        feeder.set_hub(b, p, q)
    return {"n": n_last, "rho": rho_last}

def feeder_eta(): return CFG["eta_inv"]


In [ ]:
%%writefile v2g_env_residual.py
"""
Phase-3 environment — RESIDUAL-OVER-DROOP, DISCHARGE-ONLY.

Fixes the two failure modes seen in the direct-action foresight agent:
  (1) It couldn't match droop's basic competence from a cold start.
  (2) It "rationed" by CHARGING mid-day (banking SOC), which draws grid power and
      pushes sagging voltages even lower -> it *created* violations.

Here the agent outputs a bounded RESIDUAL added to the droop setpoint:
      P = clip( droop_P + a_P * P_rated , 0 , P_rated )    # discharge-only (>= 0)
      Q = clip( droop_Q + a_Q * Q_rated , -Q_rated , Q_rated )
so a = 0  ==>  plain droop (a guaranteed performance floor), and the agent can only
learn to *withhold* discharge in the morning (save SOC) or *add* discharge in the
evening. It can never charge during the support window. The augmented state
(SOC / availability / time) lets it time this intertemporally.
"""
import numpy as np, gymnasium as gym
from gymnasium import spaces
from v2g_core import Feeder, EVFleet, reward_from_v, lam_profile, droop_pq, CFG, AVAIL


class V2GVoltEnvResidual(gym.Env):
    def __init__(self, hub_buses, cfg=CFG, peak_range=(1.0, 3.5), load_noise=0.05,
                 soc_cost=0.5, seed=0):
        super().__init__()
        self.cfg = cfg; self.peak_range = peak_range; self.load_noise = load_noise
        self.soc_cost = soc_cost
        self.fd = Feeder(hub_buses); self.nb = len(self.fd.buses); self.nh = len(hub_buses)
        self.hours = list(cfg["active_hours"]); self.H = len(self.hours)
        self.fleets = {b: EVFleet(cfg) for b in hub_buses}
        self.rng = np.random.default_rng(seed)
        self.action_space = spaces.Box(-1.0, 1.0, shape=(2 * self.nh,), dtype=np.float32)
        obs_dim = self.nb + 2 + 2 * self.nh
        self.observation_space = spaces.Box(-0.1, 2.0, shape=(obs_dim,), dtype=np.float32)
        self._ti = 0; self._peak = 1.5; self._lam_day = lam_profile(1.5)

    def _hour(self):
        return self.hours[min(self._ti, self.H - 1)]

    def _obs(self):
        h = self._hour(); lam = self._lam_day[h]
        self.fd.set_load(lam); self.fd.zero_hubs(); self.fd.solve()
        vpu = self.fd.bus_vpu()
        lam_norm = (lam - 0.1) / (4.0 - 0.1)
        hour_norm = (h - self.hours[0]) / (self.hours[-1] - self.hours[0])
        socs = [self.fleets[b].soc for b in self.fd.hub_buses]
        navs = [self.fleets[b].n_avail(h) / self.cfg["n_ev"] for b in self.fd.hub_buses]
        return np.concatenate([vpu, [lam_norm, hour_norm], socs, navs]).astype(np.float32)

    def reset(self, seed=None, options=None):
        if seed is not None:
            self.rng = np.random.default_rng(seed)
        self._peak = float(self.rng.uniform(*self.peak_range))
        shape = lam_profile(self._peak)
        noise = 1.0 + self.rng.normal(0.0, self.load_noise, size=shape.shape)
        self._lam_day = np.clip(shape * noise, 0.05, None)
        for fl in self.fleets.values():
            fl.reset()
        self._ti = 0
        return self._obs(), {}

    def _residual_setpoint(self, b, i, a):
        """droop baseline (from current raw voltage) + bounded residual; discharge-only P."""
        v = self.fd.hub_vpu(b)
        p_d, q_d = droop_pq(v, self.cfg["P_rated"], self.cfg["Q_rated"])
        p_req = float(np.clip(max(0.0, p_d) + a[2 * i]     * self.cfg["P_rated"], 0.0, self.cfg["P_rated"]))
        q_req = float(np.clip(q_d          + a[2 * i + 1] * self.cfg["Q_rated"], -self.cfg["Q_rated"], self.cfg["Q_rated"]))
        return p_req, q_req

    def step(self, action):
        a = np.clip(action, -1, 1); h = self._hour()
        discharge = 0.0
        for i, b in enumerate(self.fd.hub_buses):
            p_req, q_req = self._residual_setpoint(b, i, a)
            p_sup, q_sup, _, _ = self.fleets[b].apply(p_req, q_req, h)
            self.fd.set_hub(b, p_sup, q_sup)
            discharge += max(0.0, p_sup) / self.cfg["P_rated"]
        self.fd.solve()
        r = reward_from_v(self.fd.bus_vpu(), self.cfg["v_min"], self.cfg["v_max"])
        r -= self.soc_cost * discharge
        self._ti += 1
        trunc = self._ti >= self.H; term = False
        return self._obs(), float(r), term, trunc, {}


def evaluate_day_residual(feeder, peak, policy, cfg=CFG):
    """Deterministic daily rollout of a residual-over-droop policy. If policy is None,
    plays ZERO residual == plain (open-loop) droop, to verify the performance floor.
    Obs layout matches V2GVoltEnvResidual._obs exactly."""
    lam = lam_profile(peak); hours = list(cfg["active_hours"])
    fleets = {b: EVFleet(cfg) for b in feeder.hub_buses}
    for fl in fleets.values():
        fl.reset()
    rec = {"hour": [], "vmean": [], "soc": [], "rho": []}
    for h in hours:
        feeder.set_load(lam[h]); feeder.zero_hubs(); feeder.solve()
        vpu = feeder.bus_vpu()
        lam_norm = (lam[h] - 0.1) / (4.0 - 0.1)
        hour_norm = (h - hours[0]) / (hours[-1] - hours[0])
        socs = [fleets[b].soc for b in feeder.hub_buses]
        navs = [fleets[b].n_avail(h) / cfg["n_ev"] for b in feeder.hub_buses]
        if policy is None:
            act = np.zeros(2 * len(feeder.hub_buses), dtype=np.float32)
        else:
            obs = np.concatenate([vpu, [lam_norm, hour_norm], socs, navs]).astype(np.float32)
            act, _ = policy.predict(obs, deterministic=True)
        rho_last = 1.0
        for i, b in enumerate(feeder.hub_buses):
            v = feeder.hub_vpu(b)
            p_d, q_d = droop_pq(v, cfg["P_rated"], cfg["Q_rated"])
            p_req = float(np.clip(max(0.0, p_d) + act[2 * i]     * cfg["P_rated"], 0.0, cfg["P_rated"]))
            q_req = float(np.clip(q_d          + act[2 * i + 1] * cfg["Q_rated"], -cfg["Q_rated"], cfg["Q_rated"]))
            p_sup, q_sup, rho, _ = fleets[b].apply(p_req, q_req, h); rho_last = rho
            feeder.set_hub(b, p_sup, q_sup)
        feeder.solve()
        rec["hour"].append(h); rec["vmean"].append(feeder.feeder_mean())
        rec["soc"].append(float(np.mean([fleets[b].soc for b in feeder.hub_buses])))
        rec["rho"].append(rho_last)
    vm = np.array(rec["vmean"])
    stats = dict(Mean=round(vm.mean(), 3), Min=round(vm.min(), 3), Max=round(vm.max(), 3),
                 Viol=int((vm < cfg["v_min"]).sum()))
    return stats, rec


In [ ]:
%%writefile safe_path.py
"""
SAFE-PATH driver: single-hub, the degradation/SOC axis (the guaranteed win).

Compares, on an IDENTICAL feeder/day, two COMPLETE controllers:
  (1) closed-loop droop         -- the paper's proper baseline (damped fixed point)
  (2) residual-over-droop RL    -- P = clip(droop_P + a*P_rated, 0, P_rated), SOC/time-aware

Logs BOTH axes consistently for each:
  * voltage support : violation-hours (feeder-mean AND worst-bus), worst-bus min voltage
  * battery wear    : total energy discharged (kWh), end-of-day mean SOC

Hypothesis under test: residual-RL holds voltage support >= droop while discharging
LESS energy (higher end SOC) -- droop is memoryless, so it discharges greedily on local
voltage even for cosmetic (in-band) sags.

MEASURED OUTCOME: the hypothesis FAILS at a single hub. The worst bus never enters the
voltage band there, so there is no in-band slack to withhold and the voltage penalty
(30-100+/hr) dwarfs the SOC cost (max 0.5/hr) ~100:1 -- the optimal policy is maximal
discharge, and the agent converges to it. Because a=0 is exactly droop, the agent cannot
be handicapped, which is what makes the null informative: the binding constraint is hub
energy/siting, not control. See RESULTS.md.
"""
import sys, time, numpy as np
import torch
torch.set_num_threads(4)
import v2g_core as core
from v2g_core import Feeder, EVFleet, droop_pq, lam_profile, CFG
from v2g_env_residual import V2GVoltEnvResidual
from stable_baselines3 import SAC

BASE_AVAIL = core.AVAIL.copy()


def set_avail_scale(s):
    core.AVAIL = np.clip(BASE_AVAIL * s, 0.05, 1.0)


def rollout(feeder, peak, mode, policy=None, cfg=CFG):
    """Unified daily rollout. mode in {'droop','residual','zero'}.
    Returns (summary, rec) with consistent energy + worst-bus logging."""
    lam = lam_profile(peak); hours = cfg["active_hours"]
    fleets = {b: EVFleet(cfg) for b in feeder.hub_buses}
    for fl in fleets.values():
        fl.reset()
    rec = {"hour": [], "vmean": [], "vmin": [], "disch": [], "soc": []}
    for h in hours:
        feeder.set_load(lam[h]); feeder.zero_hubs(); feeder.solve()
        vraw = feeder.bus_vpu()
        disch = 0.0
        if mode == "droop":
            # closed-loop droop: damped fixed point with fleet-preview scaling (no commit)
            sp = {b: (0.0, 0.0) for b in feeder.hub_buses}
            damp = 0.6
            for _ in range(25):
                for b in feeder.hub_buses:
                    v = feeder.hub_vpu(b)
                    p, q = droop_pq(v, cfg["P_rated"], cfg["Q_rated"])
                    p0, q0 = sp[b]; sp[b] = ((1 - damp) * p0 + damp * p, (1 - damp) * q0 + damp * q)
                for b, (p, q) in sp.items():
                    s = np.hypot(p, q); pf = s / cfg["eta_inv"]; pa = fleets[b].avail_power(h)
                    rho = min(1.0, pa / pf) if pf > 1e-6 else 1.0
                    feeder.set_hub(b, rho * p, rho * q)
                feeder.solve()
            for b, (p, q) in sp.items():
                p_sup, q_sup, rho, n = fleets[b].apply(p, q, h)
                feeder.set_hub(b, p_sup, q_sup); disch += max(0.0, p_sup)
            feeder.solve()
        else:
            lam_norm = (lam[h] - 0.1) / (4.0 - 0.1)
            hour_norm = (h - hours[0]) / (hours[-1] - hours[0])
            socs = [fleets[b].soc for b in feeder.hub_buses]
            navs = [fleets[b].n_avail(h) / cfg["n_ev"] for b in feeder.hub_buses]
            if mode == "zero":
                act = np.zeros(2 * len(feeder.hub_buses), dtype=np.float32)
            else:
                obs = np.concatenate([vraw, [lam_norm, hour_norm], socs, navs]).astype(np.float32)
                act, _ = policy.predict(obs, deterministic=True)
            for i, b in enumerate(feeder.hub_buses):
                v = feeder.hub_vpu(b)
                p_d, q_d = droop_pq(v, cfg["P_rated"], cfg["Q_rated"])
                p_req = float(np.clip(max(0.0, p_d) + act[2 * i] * cfg["P_rated"], 0.0, cfg["P_rated"]))
                q_req = float(np.clip(q_d + act[2 * i + 1] * cfg["Q_rated"], -cfg["Q_rated"], cfg["Q_rated"]))
                p_sup, q_sup, rho, n = fleets[b].apply(p_req, q_req, h)
                feeder.set_hub(b, p_sup, q_sup); disch += max(0.0, p_sup)
            feeder.solve()
        vpu = feeder.bus_vpu()
        rec["hour"].append(h); rec["vmean"].append(feeder.feeder_mean())
        rec["vmin"].append(float(vpu.min())); rec["disch"].append(disch)
        rec["soc"].append(float(np.mean([fleets[b].soc for b in feeder.hub_buses])))
    vm = np.array(rec["vmean"]); vmn = np.array(rec["vmin"])
    summary = dict(
        ViolMean=int((vm < cfg["v_min"]).sum()),
        ViolBus=int((vmn < cfg["v_min"]).sum()),
        Vmin=round(float(vmn.min()), 3),
        Energy=round(float(sum(rec["disch"])), 1),   # kWh discharged over the day
        SOCend=round(rec["soc"][-1], 3),
    )
    return summary, rec


def train_residual(hub_buses, peak_lo, peak_hi, steps, avail_scale=1.0, soc_cost=0.5, seed=0):
    set_avail_scale(avail_scale)
    env = V2GVoltEnvResidual(hub_buses, peak_range=(peak_lo, peak_hi),
                             soc_cost=soc_cost, seed=seed)
    m = SAC("MlpPolicy", env, learning_rate=3e-4, batch_size=256, gamma=0.99,
            buffer_size=200_000, learning_starts=1000, tau=0.005,
            policy_kwargs=dict(net_arch=[256, 256]), device="cpu", seed=seed, verbose=0)
    t0 = time.time(); C = 5000; done = 0
    while done < steps:
        n = min(C, steps - done)
        m.learn(total_timesteps=n, reset_num_timesteps=(done == 0), progress_bar=False)
        done += n
        print(f"    train {done}/{steps}  {time.time()-t0:.0f}s", flush=True)
    return m


def compare(feeder, peak, policy, name, avail_scale=1.0):
    set_avail_scale(avail_scale)
    z, zrec = rollout(feeder, peak, "zero")       # floor sanity (open-loop droop, a=0)
    dr, drec = rollout(feeder, peak, "droop")     # proper closed-loop droop baseline
    rs, rrec = rollout(feeder, peak, "residual", policy=policy)
    dE = dr["Energy"] - rs["Energy"]
    pct = 100.0 * dE / dr["Energy"] if dr["Energy"] > 1e-6 else 0.0
    vtag = "OK" if rs["ViolBus"] <= dr["ViolBus"] else "WORSE"
    print(f"\n  === {name}  (peak={peak}, avail_scale={avail_scale}) ===")
    print(f"    {'controller':<16}{'ViolMean':>9}{'ViolBus':>9}{'Vmin':>7}{'Energy(kWh)':>13}{'SOCend':>8}")
    for tag, d in [("closed-droop", dr), ("residual-RL", rs), ("(a=0 floor)", z)]:
        print(f"    {tag:<16}{d['ViolMean']:>9}{d['ViolBus']:>9}{d['Vmin']:>7}{d['Energy']:>13}{d['SOCend']:>8}")
    print(f"    --> voltage support: {vtag} (worst-bus viol {rs['ViolBus']} vs droop {dr['ViolBus']})")
    print(f"    --> ENERGY SAVED   : {dE:.1f} kWh ({pct:.1f}% less discharge), "
          f"end-SOC {rs['SOCend']} vs {dr['SOCend']}")
    return dr, rs, drec, rrec


if __name__ == "__main__":
    steps = int(sys.argv[1]) if len(sys.argv) > 1 else 20000
    hub = CFG["hub_bus_single"]
    print(f"SINGLE-HUB safe-path run | residual SAC steps={steps} | hub={hub}")
    fd = Feeder(hub)
    import json
    dump = {}
    for peak, nm, tag in [(CFG["peak_mild"], "single-hub MILD", "mild"),
                          (CFG["peak_aggr"], "single-hub AGGR", "aggr")]:
        print(f"\n[train residual @ {nm}] ...", flush=True)
        # train across the peak's neighbourhood so the agent generalises around it
        m = train_residual(hub, peak * 0.8, peak * 1.2, steps=steps, avail_scale=1.0)
        m.save(f"sac_residual_single_{tag}.zip")
        dr, rs, drec, rrec = compare(fd, peak, m, nm, avail_scale=1.0)
        dump[tag] = dict(peak=peak, droop=dr, residual=rs,
                         droop_rec=drec, residual_rec=rrec)
    with open("safe_path_single.json", "w") as f:
        json.dump(dump, f, indent=1)
    print("\nsaved: sac_residual_single_{mild,aggr}.zip, safe_path_single.json")
    print("DONE.")


In [ ]:
# 1. run parameters  (bump these for a cleaner result; defaults finish in ~15-20 min on CPU)
STEPS_HEADLINE = 20000   # SAC steps for the single-hub headline agents (mild & aggr)
STEPS_SWEEP    = 8000    # SAC steps per availability level in the sweep
print("STEPS_HEADLINE =", STEPS_HEADLINE, "| STEPS_SWEEP =", STEPS_SWEEP)

## 1. Reproduction sanity — baseline vs closed-loop droop (violation-hours)
Confirms the setup matches the paper: droop cuts violation-hours vs the no-support baseline. `Viol` = hours with feeder-mean voltage < 0.95.

In [ ]:
import importlib, v2g_core; importlib.reload(v2g_core)
from v2g_core import Feeder, CFG, EVFleet, evaluate_day

print("violation-hours (feeder-mean < 0.95):  baseline  ->  closed-loop droop")
for buses, label in [(CFG["hub_bus_single"], "single"), (CFG["hub_buses_multi"], "multi ")]:
    fd = Feeder(buses); fleets = {b: EVFleet(CFG) for b in buses}
    for peak, pn in [(CFG["peak_mild"], "mild"), (CFG["peak_aggr"], "aggr")]:
        base, _ = evaluate_day(fd, peak, "baseline")
        dr, _   = evaluate_day(fd, peak, "droop", ev_constrained=True, fleets=fleets)
        print(f"  {label} {pn}:  baseline {base['Viol']:2d}  ->  droop {dr['Viol']:2d}   "
              f"(droop worst-bus Vmin {dr['Min']})")

## 2. Headline — single-hub: does foresight beat droop?
Trains the residual-over-droop agent and compares it to closed-loop droop on **both** axes:
voltage support (violation-hours, worst-bus Vmin) **and** battery wear (kWh discharged, end-of-day SOC).
`(a=0 floor)` is the zero-residual sanity row (open-loop droop) — it verifies the safety floor is wired up.

A **negative** `ENERGY SAVED` means the agent discharged *more* than droop. That is what the full-length
runs actually produce here; see `RESULTS.md` for why.


In [ ]:
import importlib, safe_path, json
importlib.reload(safe_path)
from v2g_core import Feeder, CFG

hub = CFG["hub_bus_single"]; fd = Feeder(hub)
results = {}
for peak, tag in [(CFG["peak_mild"], "mild"), (CFG["peak_aggr"], "aggr")]:
    print(f"\n===== training residual agent @ single-hub {tag.upper()} (peak={peak}) =====")
    m = safe_path.train_residual(hub, peak*0.8, peak*1.2, steps=STEPS_HEADLINE, avail_scale=1.0)
    m.save(f"sac_residual_single_{tag}.zip")
    dr, rs, drec, rrec = safe_path.compare(fd, peak, m, f"single-hub {tag.upper()}", avail_scale=1.0)
    results[tag] = dict(peak=peak, droop=dr, residual=rs, droop_rec=drec, residual_rec=rrec)
json.dump(results, open("safe_path_single.json", "w"), indent=1)
print("\nsaved safe_path_single.json + sac_residual_single_{mild,aggr}.zip")

## 3. Availability sweep — does any availability level open a gap?
Single-hub, mild load, fleet availability swept across the 40–70% window. `VBgap`/`VMgap` > 0 *would* mean
residual-RL beats droop on worst-bus / feeder-mean violation-hours; `dEnergy%` > 0 *would* mean less energy
discharged at equal-or-better support. In the measured runs neither happens at any level.


In [ ]:
import numpy as np, json
from v2g_core import Feeder, CFG
from safe_path import train_residual, rollout, set_avail_scale, BASE_AVAIL

def mean_avail(scale):
    a = np.clip(BASE_AVAIL * scale, 0.05, 1.0)
    return float(np.mean([a[h] for h in CFG["active_hours"]]))

peak = CFG["peak_mild"]; hub = CFG["hub_bus_single"]; fd = Feeder(hub)
scales = [0.6, 0.75, 0.9, 1.05, 1.2]
print(f"AVAILABILITY SWEEP | single-hub MILD peak={peak} | steps/level={STEPS_SWEEP}")
hdr = f"{'avail%':>7}{'scale':>7}{'droopVB':>9}{'rlVB':>6}{'droopVM':>9}{'rlVM':>6}{'dEnergy%':>10}{'VBgap':>7}{'VMgap':>7}"
print(hdr)
sweep_rows = []
for s in scales:
    m = train_residual(hub, peak*0.8, peak*1.2, steps=STEPS_SWEEP, avail_scale=s)
    set_avail_scale(s)
    dr, _ = rollout(fd, peak, "droop"); rs, _ = rollout(fd, peak, "residual", policy=m)
    dEpct = 100*(dr["Energy"]-rs["Energy"])/dr["Energy"] if dr["Energy"] > 1e-6 else 0.0
    vbgap = dr["ViolBus"]-rs["ViolBus"]; vmgap = dr["ViolMean"]-rs["ViolMean"]
    ma = 100*mean_avail(s)
    print(f"{ma:7.0f}{s:7.2f}{dr['ViolBus']:9d}{rs['ViolBus']:6d}{dr['ViolMean']:9d}{rs['ViolMean']:6d}{dEpct:10.1f}{vbgap:7d}{vmgap:7d}")
    sweep_rows.append(dict(avail=round(ma,1), scale=s, droop=dr, residual=rs,
                          dEpct=round(dEpct,1), vbgap=vbgap, vmgap=vmgap))
json.dump(sweep_rows, open("sweep_single.json", "w"), indent=1)
print("saved sweep_single.json")

## 4. Figures

In [ ]:
import json, numpy as np
import matplotlib.pyplot as plt
D = json.load(open("safe_path_single.json"))

def fig_degradation(tag):
    d = D[tag]; dr, rs = d["droop"], d["residual"]
    drec, rrec = d["droop_rec"], d["residual_rec"]; hrs = drec["hour"]
    fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
    ax[0].plot(hrs, drec["soc"], "o-", color="#c0392b", label=f"droop (end {dr['SOCend']})")
    ax[0].plot(hrs, rrec["soc"], "s-", color="#27ae60", label=f"residual-RL (end {rs['SOCend']})")
    ax[0].axhline(0.2, ls="--", color="grey", lw=1, label="SOC floor")
    ax[0].set_title("(A) Fleet SOC over the day"); ax[0].set_xlabel("hour"); ax[0].set_ylabel("mean SOC")
    ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)
    ax[1].plot(hrs, drec["vmean"], "o-", color="#c0392b", label=f"droop (viol {dr['ViolMean']})")
    ax[1].plot(hrs, rrec["vmean"], "s-", color="#27ae60", label=f"residual-RL (viol {rs['ViolMean']})")
    ax[1].axhline(0.95, ls="--", color="k", lw=1, label="0.95 limit")
    ax[1].set_title("(B) Feeder-mean voltage"); ax[1].set_xlabel("hour"); ax[1].set_ylabel("p.u.")
    ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3)
    ax[2].plot(hrs, np.cumsum(drec["disch"]), "o-", color="#c0392b", label=f"droop ({dr['Energy']:.0f} kWh)")
    ax[2].plot(hrs, np.cumsum(rrec["disch"]), "s-", color="#27ae60", label=f"residual-RL ({rs['Energy']:.0f} kWh)")
    ax[2].set_title("(C) Cumulative energy discharged"); ax[2].set_xlabel("hour"); ax[2].set_ylabel("kWh")
    ax[2].legend(fontsize=8); ax[2].grid(alpha=0.3)
    dE = dr["Energy"] - rs["Energy"]          # + = agent discharged less than droop
    pct = 100*dE/dr["Energy"] if dr["Energy"] > 1e-6 else 0.0
    word = "less" if dE >= 0 else "MORE"
    supp = ("equal" if rs["ViolBus"] == dr["ViolBus"]
            else "better" if rs["ViolBus"] < dr["ViolBus"] else "WORSE")
    fig.suptitle(f"Single-hub {tag.upper()}: {supp} voltage support "
                 f"(worst-bus viol {rs['ViolBus']} vs droop {dr['ViolBus']}), "
                 f"{abs(pct):.0f}% {word} battery throughput",
                 fontsize=12, y=1.02)
    fig.tight_layout(); fig.savefig(f"fig_degradation_{tag}.png", dpi=110, bbox_inches="tight")
    plt.show()

for t in ["mild", "aggr"]:
    fig_degradation(t)

# sweep figure
try:
    S = json.load(open("sweep_single.json"))
    av = [r["avail"] for r in S]
    fig, ax1 = plt.subplots(figsize=(7, 4.5))
    ax1.plot(av, [r["droop"]["ViolMean"] for r in S], "o-", color="#c0392b", label="droop viol-hours")
    ax1.plot(av, [r["residual"]["ViolMean"] for r in S], "s-", color="#27ae60", label="residual-RL viol-hours")
    ax1.set_xlabel("mean fleet availability (%)"); ax1.set_ylabel("feeder-mean violation-hours")
    ax1.legend(loc="upper right", fontsize=8); ax1.grid(alpha=0.3)
    ax2 = ax1.twinx(); ax2.bar(av, [r["dEpct"] for r in S], width=2.5, alpha=0.18, color="#2980b9")
    ax2.set_ylabel("energy vs droop (%):  + saved,  − extra", color="#2980b9")
    ax1.set_title("Availability sweep (single-hub mild): does foresight open a gap?")
    fig.tight_layout(); fig.savefig("fig_sweep.png", dpi=110, bbox_inches="tight"); plt.show()
except FileNotFoundError:
    print("run the sweep cell first for fig_sweep.png")

## What to send back
Copy back:
1. **Reproduction table** (cell 1) — baseline vs droop violation-hours.
2. **Headline tables** (cell 2) — both `=== single-hub MILD/AGGR ===` blocks plus the `ENERGY SAVED` lines.
3. **Sweep table** (cell 3).
4. The PNGs: `fig_degradation_mild.png`, `fig_degradation_aggr.png`, `fig_sweep.png`.

**What the runs so far show** (two independent 20k-step runs — see `RESULTS.md` for the tables):
single-hub, residual-RL vs droop → **no win on either axis**. Worst-bus violation-hours tie at mild
(15 vs 15) and are *worse* at aggressive (18 vs 17); energy discharged came out **+20% higher** in one run
and 2% lower in the other, i.e. the run-to-run spread is larger than any effect. The availability sweep
finds no level where a gap opens.

This is a real result, not a failed experiment: with the droop floor wired in (`a=0` ≡ droop) the agent
*cannot* be handicapped, so "no better than droop" localizes the bottleneck in the physics — one hub's
fleet holds roughly an hour of full-power support against a multi-hour deficit — rather than in the
controller. Longer training (`STEPS_HEADLINE=40000`) sharpens the estimate; it does not change the sign.
